# 1.3 VAF and alt-depth QC graphs

This notebook summarizes per-variant VAF (`Sample.AltFrac`) and alternate-read depth (`Sample.AltDepth`) from the on/off-target QC table.

One version for all variants (`FILTER == "PASS"`) and the other for the 16 target breast-cancer genes (`on_target == True`).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import numpy as np

run_label = "Run2"
input_csv = Path("/home/donetski/Notebooks/OutputFiles/04_qc_checking_on_target/04_Run2_per_variant_target_status_FULL.csv")
output_dir = Path("/home/donetski/Notebooks/OutputFiles/1.3_vaf_density_graphs_KDE") / run_label.lower()
output_prefix = "1.3"

filter_to_pass = True
filter_to_on_target = True
callers = ["DeepVariant", "Mutect2"]

#graph_version = "vaf_0p01_0p03_0p05_0p07_0p10"
#vaf_thresholds = [0.01, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20]

graph_version = "vaf_0p05_0p10_0p15_0p20"
vaf_thresholds = [0.05, 0.10,

#Sample.AltDepth >= threshold
altdepth_thresholds = [2, 3, 5, 7, 10]
vaf_thresholds = [0.01, 0.03, 0.05, 0.07, 0.10]

TARGET_GENES = [
    "ATM", "BARD1", "BRCA1", "BRCA2", "CDH1", "CDKN2A", "CHEK2", "MLH1",
    "MSH2", "MSH6", "PALB2", "PMS2", "PTEN", "RAD51C", "RAD51D", "TP53",
]

output_dir.mkdir(parents=True, exist_ok=True)

## Load and QC-filter the variant table

The numeric VAF/depth columns are coerced to numbers before plotting. Rows with nonnumeric values become missing for those specific plotting fields.

In [ ]:
required_columns = ["caller", "Gene", "Sample.AltFrac", "Sample.AltDepth", "Sample.Depth"]

df = pd.read_csv(input_csv, low_memory=False)
numeric_columns = ["Sample.AltFrac", "Sample.AltDepth", "Sample.Depth"]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

# These rows have no alternate-read support (alt.depth = 0), so exclude them
df_plot = df_qc[df_qc["Sample.AltDepth"] >= 1].copy()
print("Rows excluded because AltDepth = 0:", len(df_qc) - len(df_plot))
df_qc = df.copy()

if filter_to_pass:
    required_columns.append("FILTER")
    df_qc = df_qc[df_qc["FILTER"].eq("PASS")].copy()

if filter_to_on_target:
    required_columns.append("on_target")
    on_target_mask = df_qc["on_target"].astype(str).str.lower().isin(["true", "1", "yes"])
    df_qc = df_qc[on_target_mask].copy()

print(f"Input rows: {len(df):,}")
print(f"Rows after base QC filters: {len(df_qc):,}")
print(df_qc["caller"].value_counts(dropna=False))

## Create the two analysis sets

The target-gene set uses the same 16-gene list as notebook `1.2`. The all-variant set is not gene-restricted.

In [ ]:
target_gene_mask = df_plot["Gene"].astype(str).str.upper().isin(TARGET_GENES)
analysis_sets = {"all_variants": df_plot.copy(), "target_16_genes": df_plot[target_gene_mask].copy() }

for set_name, set_df in analysis_sets.items():
    set_output_dir = output_dir / set_name
    set_output_dir.mkdir(parents=True, exist_ok=True)

    out_csv = set_output_dir / f"{output_prefix}_{run_label}_{set_name}_qc_variants.csv"
    set_df.to_csv(out_csv, index=False)

    print(f"{set_name}: {len(set_df):,} rows")
    print("Saved:", out_csv)

## Plotting helper functions

In [ ]:
def safe_name(value):
    return str(value).replace(" ", "_").replace("/", "_").replace(">=", "ge").replace(">", "gt").lower()


def format_threshold(value):
    return str(value).replace(".", "p")

def plot_gaussian_kde(series, label, x_grid):
    values = pd.to_numeric(series,errors="coerce").dropna()
    # gaussian_kde needs at least two non-identical values.
    if len(values) < 2 or values.nunique() < 2:
        return False

    # Fit explicit Gaussian KDE using every remaining value.
    kde = gaussian_kde(values)
    
    # Evaluate and draw the KDE on the shared x-axis positions.
    plt.plot(x_grid, kde(x_grid), label=label)
    return True

def plot_single_caller_vaf_density(caller_df, caller_name, set_name, figure_dir):
    plt.figure(figsize=(8, 5))
    plotted = plot_density(caller_df["Sample.AltFrac"], f"{caller_name} (n={len(caller_df):,})")

    if not plotted:
        plt.close()
        print(f"Skipped VAF density for {set_name} / {caller_name}: not enough values")
        return

    plt.axvline(0.5, linestyle="--", label="VAF ~0.5")
    plt.axvline(1.0, linestyle="--", label="VAF ~1.0")
    plt.xlim(0, 1.05)
    plt.title(f"{run_label} {set_name}: {caller_name} VAF density")
    plt.xlabel("Sample.AltFrac")
    plt.ylabel("Density")
    plt.legend()

    caller_file_name = safe_name(caller_name)
    out_png = figure_dir / f"{output_prefix}_{run_label}_{set_name}_{caller_file_name}_vaf_density.png"
    save_plot(out_png)


def plot_altdepth_histogram(caller_df, caller_name, set_name, figure_dir):
    values = caller_df["Sample.AltDepth"].dropna()
    if values.empty:
        print(f"Skipped AltDepth histogram for {set_name} / {caller_name}: no values")
        return

    plt.figure(figsize=(8, 5))
    values.plot(kind="hist", bins=50)
    plt.title(f"{run_label} {set_name}: {caller_name} AltDepth distribution")
    plt.xlabel("Sample.AltDepth")
    plt.ylabel("Number of variant calls")

    caller_file_name = safe_name(caller_name)
    out_png = figure_dir / f"{output_prefix}_{run_label}_{set_name}_{caller_file_name}_altdepth_histogram.png"
    save_plot(out_png)


def plot_altdepth_zoom(caller_df, caller_name, set_name, figure_dir):
    zoom_df = caller_df[caller_df["Sample.AltDepth"].between(1, 20)].copy()
    if zoom_df.empty:
        print(f"Skipped AltDepth 1-20 histogram for {set_name} / {caller_name}: no rows")
        return

    plt.figure(figsize=(8, 5))
    plt.hist(zoom_df["Sample.AltDepth"].dropna(), bins=range(1, 22), align="left")
    plt.title(f"{run_label} {set_name}: {caller_name} AltDepth distribution (1-20)")
    plt.xlabel("Sample.AltDepth")
    plt.ylabel("Number of variant calls")
    plt.xticks(range(1, 21))

    caller_file_name = safe_name(caller_name)
    out_png = figure_dir / f"{output_prefix}_{run_label}_{set_name}_{caller_file_name}_altdepth_1_to_20.png"
    save_plot(out_png)

def plot_vaf_kde_by_altdepth(caller_df, caller_name, set_name, figure_dir):
    groups = [("All valid AltDepth", caller_df)]

    for threshold in altdepth_thresholds:
        subset = caller_df[caller_df["Sample.AltDepth"] >= threshold]
        groups.append((f"AltDepth >= {threshold}", subset))

    x_grid = np.linspace(0, 1.05, 300)
    plt.figure(figsize=(8, 5))
    plotted_any = False

    for label, subset in groups:
        plotted_any |= plot_gaussian_kde(
            subset["Sample.AltFrac"],
            f"{label} (n={len(subset):,})",
            x_grid,
        )

    if not plotted_any:
        plt.close()
        print(f"Skipped VAF KDE for {set_name} / {caller_name}: not enough values")
        return

    plt.axvline(0.5, linestyle="--", label="VAF ~0.5")
    plt.axvline(1.0, linestyle="--", label="VAF ~1.0")
    plt.xlim(0, 1.05)
    plt.xlabel("Sample.AltFrac")
    plt.ylabel("Density")
    plt.title(f"{run_label} {set_name}: {caller_name} VAF Gaussian KDE by AltDepth threshold")
    plt.legend()

    caller_file_name = safe_name(caller_name)
    out_png = figure_dir / (
        f"{output_prefix}_{run_label}_{set_name}_{caller_file_name}"
        "_vaf_gaussian_kde_by_altdepth_thresholds.png"
    )
    save_plot(out_png)

def plot_altdepth_kde_by_vaf(caller_df, caller_name, set_name, figure_dir):
    groups = [("All VAF", caller_df)]

    for threshold in vaf_thresholds:
        subset = caller_df[caller_df["Sample.AltFrac"] >= threshold]
        groups.append((f"VAF >= {threshold}", subset))

    x_grid = np.linspace(1, 20, 500)
    all_altdepth = caller_df["Sample.AltDepth"].dropna()
    x_grid = np.linspace(all_altdepth.min(),all_altdepth.max(),500)

    plt.figure(figsize=(8, 5))
    plotted_any = False

    for label, subset in groups:
        plotted_any |= plot_gaussian_kde(
            subset["Sample.AltDepth"], f"{label} (n={len(subset):,})", x_grid
        )

    if not plotted_any:
        plt.close()
        print(f"Skipped AltDepth KDE for {set_name} / {caller_name}: not enough values")
        return

    plt.xlim(1, 100)
    plt.xticks(range(0, 101, 5))

    plt.xlabel("Sample.AltDepth")
    plt.ylabel("Density")
    plt.title(f"{run_label} {set_name}: {caller_name} AltDepth Gaussian KDE by VAF threshold")
    plt.legend()

    caller_file_name = safe_name(caller_name)
    out_png = figure_dir / f"{output_prefix}_{run_label}_{set_name}_{caller_file_name}_altdepth_gaussian_kde_by_vaf.png"
    save_plot(out_png)
    
def plot_vaf_density_by_altdepth(caller_df, caller_name, set_name, figure_dir):
    groups = [("All AltDepth", caller_df)]
    for threshold in altdepth_thresholds:
        groups.append((
            f"AltDepth >= {threshold}",
            caller_df[caller_df["Sample.AltDepth"] >= threshold],
        ))

    plt.figure(figsize=(8, 5))
    plotted_any = False
    for label, subset in groups:
        plotted_any |= plot_density(subset["Sample.AltFrac"], f"{label} (n={len(subset):,})")

    if not plotted_any:
        plt.close()
        print(f"Skipped VAF density by AltDepth threshold for {set_name} / {caller_name}: not enough values")
        return

    plt.axvline(0.5, linestyle="--", label="VAF ~0.5")
    plt.axvline(1.0, linestyle="--", label="VAF ~1.0")
    plt.xlim(0, 1.05)
    plt.title(f"{run_label} {set_name}: {caller_name} VAF density by AltDepth threshold")
    plt.xlabel("Sample.AltFrac")
    plt.ylabel("Density of variant calls")
    plt.legend()

    caller_file_name = safe_name(caller_name)
    out_png = figure_dir / f"{output_prefix}_{run_label}_{set_name}_{caller_file_name}_vaf_density_by_altdepth_thresholds.png"
    save_plot(out_png)


def plot_altdepth_zoom_by_vaf(caller_df, caller_name, set_name, figure_dir):
    groups = [("All VAF", caller_df)]
    for threshold in vaf_thresholds:
        groups.append((
            f"VAF >= {threshold}",
            caller_df[caller_df["Sample.AltFrac"] >= threshold],
        ))

    plt.figure(figsize=(8, 5))
    plotted_any = False
    for label, subset in groups:
        values = subset.loc[subset["Sample.AltDepth"].between(1, 20), "Sample.AltDepth"].dropna()
        if values.empty:
            continue
        plt.hist(values, bins=range(1, 22), align="left", histtype="step", linewidth=1.5, label=f"{label} (n={len(subset):,})")
        plotted_any = True

    if not plotted_any:
        plt.close()
        print(f"Skipped AltDepth by VAF threshold for {set_name} / {caller_name}: no values")
        return

    plt.title(f"{run_label} {set_name}: {caller_name} AltDepth 1-20 by VAF threshold")
    plt.xlabel("Sample.AltDepth")
    plt.ylabel("Number of variant calls")
    plt.xticks(range(1, 21))
    plt.legend()

    caller_file_name = safe_name(caller_name)
    out_png = figure_dir / f"{output_prefix}_{run_label}_{set_name}_{caller_file_name}_altdepth_1_to_20_by_vaf_thresholds.png"
    save_plot(out_png)


def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.show()
    print("Saved:", path)


## Main VAF and AltDepth outputs

For each analysis set and caller, this saves the original VAF/AltDepth plots plus two independent exploratory threshold plots:

1. VAF density across AltDepth thresholds.
2. AltDepth zoomed histograms across VAF thresholds.

These thresholds are not combined here; this notebook is only for choosing reasonable ranges.

In [ ]:
summary_rows = []

for set_name, set_df in analysis_sets.items():
    set_output_dir = output_dir / set_name
    figure_dir = output_dir / set_name / "figures" / graph_version
    figure_dir.mkdir(parents=True, exist_ok=True)

    for caller_name in callers:
        caller_df = set_df[set_df["caller"].eq(caller_name)].copy()

        # Previous non-Gaussian plots
        # plot_single_caller_vaf_density(caller_df, caller_name, set_name, figure_dir)
        # plot_altdepth_histogram(caller_df, caller_name, set_name, figure_dir)
        # plot_altdepth_zoom(caller_df, caller_name, set_name, figure_dir)
        # plot_altdepth_zoom_by_vaf(caller_df, caller_name, set_name, figure_dir)

        # SciPy Gaussian KDE plots
        plot_vaf_kde_by_altdepth(caller_df, caller_name, set_name, figure_dir)
        plot_altdepth_kde_by_vaf(caller_df, caller_name, set_name, figure_dir)

        threshold_groups = [("none", "All", None, caller_df)]

        for threshold in altdepth_thresholds:
            threshold_groups.append((
                "alt_depth",
                f"AltDepth >= {threshold}",
                threshold,
                caller_df[caller_df["Sample.AltDepth"] >= threshold],
            ))

        for threshold in vaf_thresholds:
            threshold_groups.append((
                "vaf",
                f"VAF >= {threshold}",
                threshold,
                caller_df[caller_df["Sample.AltFrac"] >= threshold],
            ))

        for threshold_type, threshold_label, threshold_value, subset in threshold_groups:
            summary_rows.append({
                "analysis_set": set_name,
                "caller": caller_name,
                "threshold_type": threshold_type,
                "threshold_label": threshold_label,
                "threshold_value": threshold_value,
                "n_variants": len(subset),
                "percent_retained": len(subset) / len(caller_df) * 100 if len(caller_df) > 0 else None,
                "min_AltDepth": subset["Sample.AltDepth"].min(),
                "median_AltDepth": subset["Sample.AltDepth"].median(),
                "median_VAF": subset["Sample.AltFrac"].median(),
            })

summary_df = pd.DataFrame(summary_rows)
summary_path = output_dir / f"{output_prefix}_{run_label}_vaf_altdepth_independent_threshold_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)
summary_df

## Mutect2 AltDepth = 1 checks

These checks focus on Mutect2 calls supported by exactly one alternate read. They are useful for seeing whether the low-alt-depth calls have unusual VAF/depth behavior and whether any are also present in DeepVariant.

In [ ]:
overlap_key_columns = ["Sample.ID", "Chr", "Start", "REF", "ALT"]
window_rows = []

for set_name, set_df in analysis_sets.items():
    set_output_dir = output_dir / set_name
    figure_dir = output_dir / set_name / "figures" / graph_version

    deepvariant_df = set_df[set_df["caller"].eq("DeepVariant")].copy()
    mutect2_df = set_df[set_df["caller"].eq("Mutect2")].copy()
    mutect2_alt1 = mutect2_df[mutect2_df["Sample.AltDepth"].eq(1)].copy()

    alt1_csv = set_output_dir / f"{output_prefix}_{run_label}_{set_name}_mutect2_altdepth_eq_1_variants.csv"
    mutect2_alt1.to_csv(alt1_csv, index=False)
    print(f"{set_name}: Mutect2 AltDepth == 1 rows: {len(mutect2_alt1):,}")
    print("Saved:", alt1_csv)

    near_05 = mutect2_alt1[mutect2_alt1["Sample.AltFrac"].between(0.45, 0.55)]
    near_10 = mutect2_alt1[mutect2_alt1["Sample.AltFrac"].between(0.95, 1.00)]
    window_rows.extend([
        {"analysis_set": set_name, "window": "0.45 <= VAF <= 0.55", "n_variants": len(near_05)},
        {"analysis_set": set_name, "window": "0.95 <= VAF <= 1.00", "n_variants": len(near_10)},
    ])

    plt.figure(figsize=(8, 5))
    plotted = plot_density(mutect2_alt1["Sample.AltFrac"], f"AltDepth = 1 (n={len(mutect2_alt1):,})")
    if plotted:
        plt.axvline(0.5, linestyle="--", label="VAF ~0.5")
        plt.axvline(1.0, linestyle="--", label="VAF ~1.0")
        plt.xlim(0, 1.05)
        plt.title(f"{run_label} {set_name}: Mutect2 VAF density for AltDepth = 1")
        plt.xlabel("Sample.AltFrac")
        plt.ylabel("Density")
        plt.legend()
        out_png = figure_dir / f"{output_prefix}_{run_label}_{set_name}_mutect2_altdepth_eq_1_vaf_density.png"
        save_plot(out_png)
    else:
        plt.close()
        print(f"Skipped Mutect2 AltDepth=1 VAF density for {set_name}: not enough values")

    if not mutect2_alt1.empty:
        plt.figure(figsize=(8, 5))
        mutect2_alt1["Sample.AltFrac"].dropna().plot(kind="hist", bins=50)
        plt.axvline(0.5, linestyle="--", label="VAF ~0.5")
        plt.axvline(1.0, linestyle="--", label="VAF ~1.0")
        plt.title(f"{run_label} {set_name}: Mutect2 VAF histogram for AltDepth = 1")
        plt.xlabel("Sample.AltFrac")
        plt.ylabel("Number of variant calls")
        plt.legend()
        out_png = figure_dir / f"{output_prefix}_{run_label}_{set_name}_mutect2_altdepth_eq_1_vaf_histogram.png"
        save_plot(out_png)

        plt.figure(figsize=(8, 5))
        mutect2_alt1["Sample.Depth"].dropna().plot(kind="hist", bins=50)
        plt.title(f"{run_label} {set_name}: Mutect2 total depth for AltDepth = 1")
        plt.xlabel("Sample.Depth")
        plt.ylabel("Number of variant calls")
        out_png = figure_dir / f"{output_prefix}_{run_label}_{set_name}_mutect2_altdepth_eq_1_total_depth_histogram.png"
        save_plot(out_png)

    missing_overlap_cols = [column for column in overlap_key_columns if column not in set_df.columns]
    if missing_overlap_cols:
        print(f"Skipped DeepVariant overlap for {set_name}; missing columns: {missing_overlap_cols}")
        continue

    overlap_alt1_dv = mutect2_alt1.merge(
        deepvariant_df,
        on=overlap_key_columns,
        suffixes=("_Mutect2", "_DeepVariant"),
        how="inner",
    )
    overlap_csv = set_output_dir / f"{output_prefix}_{run_label}_{set_name}_mutect2_altdepth_eq_1_overlap_deepvariant.csv"
    overlap_alt1_dv.to_csv(overlap_csv, index=False)
    print(f"{set_name}: Mutect2 AltDepth=1 variants also found in DeepVariant: {len(overlap_alt1_dv):,}")
    print("Saved:", overlap_csv)

window_summary_df = pd.DataFrame(window_rows)
window_summary_path = output_dir / f"{output_prefix}_{run_label}_mutect2_altdepth_eq_1_vaf_window_counts.csv"
window_summary_df.to_csv(window_summary_path, index=False)

print("Saved:", window_summary_path)
window_summary_df

In [ ]:
#Saving alt.depth = 0
zero_altdepth = df[
    pd.to_numeric(df["Sample.AltDepth"], errors="coerce").eq(0)
].copy()

zero_altdepth_csv = output_dir / f"{run_label}_AltDepth_0_rows.csv"

zero_altdepth.to_csv(zero_altdepth_csv, index=False)

print("Rows saved:", len(zero_altdepth))
print("Saved:", zero_altdepth_csv)